# Lesson 0: Find your Verdi's serial port

To connect you need three settings: the serial **port**, the laser **model**
and the **baud rate**. This lesson finds the port by asking your operating
system which serial ports exist, and shows where to read the other two.

**This notebook sends nothing.** It reads the operating system's device list.
It never opens a port or talks to any instrument.

Run the cells one at a time, top to bottom (Shift+Enter). Steps 2 and 3 ask
you to unplug and replug the adapter before running the cell.

## Why not search automatically?

It is tempting to send a query to every port and see which one answers like
a Verdi. This package deliberately does not:

- Other ports may belong to other instruments, such as signal generators,
  stages or controllers. Bytes meant for the Verdi can confuse or command them.
- Opening a port that another program is using can disturb that program.
- Silence proves nothing: a wrong baud rate, a wrong cable or a switched-off
  power supply look the same as "not a Verdi".
- The model and baud rate cannot be discovered over the link anyway.

So you identify the port from operating-system information here, then
confirm it with a single query on that one port in lesson 1.

## 1. List the serial ports

`comports()` reads the operating system's device list. It does not open any
port.

In [1]:
from serial.tools import list_ports

for port in list_ports.comports():
    print(port.device, "|", port.description, "|", port.hwid)

COM9 | USB Serial Port (COM9) | USB VID:PID=0403:6001 SER=FTB8W4Y7A
COM6 | Windfreak Serial USB Devices (COM6) | USB VID:PID=0483:A3E5 SER=209233635232 LOCATION=1-7.3.3.1.4
COM10 | USB Serial Port (COM10) | USB VID:PID=0403:6001 SER=FTB8VKKUA
COM5 | USB Serial Port (COM5) | USB VID:PID=0403:6001 SER=FTB8VN29A


Each line shows the port name (what you will enter as `PORT`), the driver's
description and a hardware ID with the USB vendor/product IDs and serial
number. Several identical "USB Serial Port" lines are common in a lab with
several adapters, so the next two steps tell them apart.

## 2. Unplug the adapter

Close any program that uses the Verdi's adapter, such as the dashboard, a
terminal program or another notebook. Then unplug the adapter's **USB plug**
from the computer and leave its RS-232 end on the laser power supply.
Unplugging the computer side does not change the laser's state.

Then run:

In [2]:
without_adapter = {port.device for port in list_ports.comports()}
sorted(without_adapter)

['COM10', 'COM3', 'COM6', 'COM9']

## 3. Plug it back in

Plug the USB plug back in, wait a few seconds for the driver, then run:

In [3]:
added = [port for port in list_ports.comports() if port.device not in without_adapter]
for port in added:
    print("PORT =", repr(port.device))
    print("  description:", port.description)
    print("  USB serial number:", port.serial_number)
if len(added) != 1:
    print(f"Expected exactly one new port, found {len(added)}.")

PORT = 'COM5'
  description: USB Serial Port (COM5)
  USB serial number: FTB8VN29A


- **One new port:** that is your `PORT`.
- **None:** wait a little longer and run the cell again. If it is still
  missing, check the adapter driver (Device Manager → Ports on Windows,
  `dmesg` on Linux).
- **Several:** another device appeared at the same time. Repeat steps 2 and 3.

The description does not show whether the adapter is RS-232 or TTL level.
Check the adapter itself: the Verdi needs RS-232 (manual Figure 5-1).

## 4. Find the same adapter next time

Port numbers can change when an adapter is plugged into another USB socket
or in a different order. Many adapters, such as FTDI ones, report a unique
USB serial number. Label the adapter with it, then look up its current port:

In [4]:
ADAPTER_SERIAL = "FTB8VN29A"  # The USB serial number printed in step 3, as a string

In [5]:
if ADAPTER_SERIAL is None:
    print("Enter ADAPTER_SERIAL in the cell above to look up its port.")
else:
    print([port.device for port in list_ports.comports() if port.serial_number == ADAPTER_SERIAL])

['COM5']


On Linux, the `/dev/serial/by-id/...` path contains the serial number and
stays the same; you can use that path directly as `PORT`.

## 5. Model and baud rate

Neither can be read over the link until you know them:

- **`MODEL`**: read `V2`, `V5` or `V6` from the laser's labels.
- **`BAUDRATE`**: read it from the power supply's front-panel menu (manual
  section 5). `19200` is only the factory default. Do not try baud rates one
  after another.

Have the operator confirm both.

## Next

Enter `PORT`, `MODEL` and `BAUDRATE` in section 1 of
[lesson 1](01_connect.ipynb). It confirms the link with a single `?SV` query
on that port only.